# Fine-Tuning the BGE-M3 Embedder for English to German Clinical Retrieval

The retrieval stage is cross-lingual: the question arrives in English while the AWMF knowledge base
is written in German. Section 4.8 showed that monolingual medical embedders collapse on this task
and that the multilingual BAAI/bge-m3 is the only viable base. This notebook examines whether
bge-m3 can be sharpened for English to German clinical alignment through contrastive fine-tuning on
the project's parallel data.

Training pairs are drawn only from the held-out benchmark rows, the 715 minus the 200-question
evaluation subset, restricted to the ten target conditions with the same disease filter as notebook
00. The embedder is trained locally with sentence-transformers. Two evaluations follow. The first
is a label-free cross-lingual proxy (English question to its German counterpart) on the untouched
200-question set. The second reproduces the metrics the chapter reports for embedders, RAGAS
Context Precision and Context Recall over the retrieved AWMF chunks, re-embedding the Neon corpus
with each embedder and scoring against the independent corpus-grounded ground truth of notebook 22.
Only the RAGAS judge uses an external API.

## 1. Environment

The notebook targets a Colab GPU runtime. Two Colab secrets are required, as in notebook 22:
NEON_DATABASE_URL for the read-only corpus, and OPENROUTER_API_KEY for the RAGAS judge only.

In [ ]:
%%capture
!pip install -q -U sentence-transformers datasets accelerate faiss-cpu
!pip install -q ragas langchain langchain-openai langchain-huggingface psycopg2-binary pgvector langchain-postgres rank_bm25 sqlalchemy nest_asyncio

## 2. Configuration

In [ ]:
CONFIG = {
    "BENCH_715":   "GP_Top10_Bilingual_Open_EndedQ.csv",
    "EVAL_200":    None,                                           # None: prefer the AI-filtered set, else the final set
    "GT_FILE":     "AWMF_CorpusGrounded_GT_Independent.csv",

    "Q_EN":  "English_Open_Question",
    "Q_DE":  "German_Open_Question",
    "GT_Q":  "English_Open_Question",
    "GT_REF": "corpus_ground_truth",

    "COLLECTION": "awmf_baseline_bge",
    "BASE_MODEL": "BAAI/bge-m3",
    "RERANKER":   "BAAI/bge-reranker-v2-m3",
    "OUTPUT_DIR": "bge-m3-clinical-ende",

    "BATCH":  16,     # 8 on a T4 if memory is tight
    "EPOCHS": 2,
    "LR":     2e-5,

    "K_DENSE": 50, "K_BM25": 50, "RRF_K": 60, "K_FINAL": 8,
    "JUDGE_MODEL": "anthropic/claude-sonnet-4.6",
}

## 3. Parallel English/German pairs from the held-out, in-scope split

The 200 evaluation questions are separated first so they serve as an untouched test set. The
training pairs are the held-out remainder restricted to the ten target conditions.

In [ ]:
import glob
import pandas as pd
from google.colab import drive, userdata
drive.mount('/content/drive')

def find(name):
    hits = glob.glob(f"/content/drive/MyDrive/**/{name}", recursive=True)
    return hits[0] if hits else name

bench = pd.read_csv(find(CONFIG["BENCH_715"]))
eval_path = None
for cand in (CONFIG["EVAL_200"], "AWMF_Golden_Dataset_200Q_AIFiltered.csv", "AWMF_Golden_Dataset_200Q_Final.csv"):
    if cand:
        h = glob.glob(f"/content/drive/MyDrive/**/{cand}", recursive=True)
        if h:
            eval_path = h[0]; break
eval_df = pd.read_csv(eval_path)
print("715 rows:", len(bench), "| 200 rows:", len(eval_df), "| columns:", list(bench.columns))

Q_EN, Q_DE = CONFIG["Q_EN"], CONFIG["Q_DE"]
eval_keys = set(eval_df[Q_EN].astype(str).str.strip())
held = bench[~bench[Q_EN].astype(str).str.strip().isin(eval_keys)].copy()

target_diseases = ["Hypertension","Diabetes","Asthma","COPD","Coronary Heart Disease",
                   "Heart Failure","Chronic Back Pain","Depression","Osteoarthritis","Dementia"]
disease_keywords = {
    "Hypertension": ['hypertension','hypertensive','blood pressure is 180','blood pressure is 160','blood pressure is 170'],
    "Diabetes": ['diabetes','hba1c','diabetic'],
    "Asthma": ['asthma','asthmatic'],
    "COPD": ['copd','chronic obstructive','emphysema'],
    "Coronary Heart Disease": ['coronary','myocardial infarction','angina'],
    "Heart Failure": ['heart failure','chf','cardiac failure','heart failing'],
    "Chronic Back Pain": ['back pain','lumbar','sciatica'],
    "Depression": ['depression','depressed','major depressive'],
    "Osteoarthritis": ['osteoarthritis','joint pain'],
    "Dementia": ['dementia','alzheimer','memory loss','cognitive decline','forgetful'],
}
def get_disease(text):
    text = str(text).lower()
    for disease, kws in disease_keywords.items():
        if any(kw in text for kw in kws):
            return disease
    return "Other"

held["Disease"] = held[Q_EN].apply(get_disease)
held = held[held["Disease"] != "Other"].dropna(subset=[Q_EN, Q_DE])
print("held-out in-scope training rows:", len(held))

## 4. Contrastive fine-tuning with in-batch negatives

MultipleNegativesRankingLoss treats each English question as an anchor and its German counterpart
as the single positive, with every other German question in the batch acting as an implicit
negative. Minimising this loss draws each English query towards its own German translation and away
from unrelated German text. The learning rate is small and the run short, because bge-m3 is already
competent and the aim is a gentle adjustment rather than retraining from scratch.

In [ ]:
from sentence_transformers import SentenceTransformer, InputExample, losses
from torch.utils.data import DataLoader

model = SentenceTransformer(CONFIG["BASE_MODEL"], device="cuda")

pairs = list(zip(held[Q_EN].astype(str), held[Q_DE].astype(str)))
examples = [InputExample(texts=[en, de]) for en, de in pairs]
loader = DataLoader(examples, shuffle=True, batch_size=CONFIG["BATCH"])
loss = losses.MultipleNegativesRankingLoss(model)

model.fit(train_objectives=[(loader, loss)], epochs=CONFIG["EPOCHS"],
          warmup_steps=max(1, int(0.1 * len(loader))), optimizer_params={"lr": CONFIG["LR"]},
          use_amp=True, show_progress_bar=True)
model.save(CONFIG["OUTPUT_DIR"])
print("saved fine-tuned embedder to", CONFIG["OUTPUT_DIR"])

## 5. Cross-lingual retrieval proxy (English question to German counterpart)

For each English question on the untouched 200-question set, the model ranks that question's German
counterpart among all German candidates. This is a label-free alignment proxy. P@1, R@5, and MRR
are reported for the base and fine-tuned models under identical conditions.

In [ ]:
import numpy as np
from sentence_transformers import SentenceTransformer

queries_en = eval_df[Q_EN].astype(str).tolist()
corpus_de  = eval_df[Q_DE].astype(str).tolist()

def xling_scores(m):
    qv = m.encode(queries_en, normalize_embeddings=True, batch_size=32)
    cv = m.encode(corpus_de,  normalize_embeddings=True, batch_size=32)
    sims = qv @ cv.T
    ranks = np.array([int(np.where(np.argsort(-row) == i)[0][0]) for i, row in enumerate(sims)])
    return {"P@1": round(float((ranks == 0).mean()), 3),
            "R@5": round(float((ranks < 5).mean()), 3),
            "MRR": round(float((1.0 / (ranks + 1)).mean()), 3)}

base_model  = SentenceTransformer(CONFIG["BASE_MODEL"], device="cuda")
tuned_model = SentenceTransformer(CONFIG["OUTPUT_DIR"], device="cuda")

results = pd.DataFrame([
    {"model": "bge-m3 (base)",       **xling_scores(base_model)},
    {"model": "bge-m3 (fine-tuned)", **xling_scores(tuned_model)},
])
results

## 6. RAGAS retrieval evaluation (comparable to Chapter 4)

The metrics the chapter reports for embedders are RAGAS Context Precision and Context Recall over
the retrieved AWMF chunks. Because the stored Neon vectors were produced by the base embedder, the
fine-tuned embedder is evaluated by re-embedding the corpus locally and building an in-memory index,
then running the same hybrid retrieval as notebook 22 (dense + BM25, fused with Reciprocal Rank
Fusion, reranked, tightened). The reference answers are the independent corpus-grounded ground truth
of notebook 22, and the only external call is the judge.

In [ ]:
import os, torch, faiss
from sqlalchemy import create_engine, text
from rank_bm25 import BM25Okapi
from sentence_transformers import CrossEncoder

NEON = userdata.get("NEON_DATABASE_URL")
os.environ["OPENROUTER_API_KEY"] = userdata.get("OPENROUTER_API_KEY")

engine = create_engine(NEON, pool_pre_ping=True)
with engine.connect() as conn:
    all_texts = [r[0] for r in conn.execute(text(
        "SELECT document FROM langchain_pg_embedding "
        "WHERE collection_id = (SELECT uuid FROM langchain_pg_collection WHERE name = :c)"),
        {"c": CONFIG["COLLECTION"]})]
bm25 = BM25Okapi([t.lower().split(" ") for t in all_texts])
_device = "cuda" if torch.cuda.is_available() else "cpu"
reranker = CrossEncoder(CONFIG["RERANKER"], max_length=512, device=_device)
print("corpus chunks:", len(all_texts))

K_DENSE, K_BM25, RRF_K, K_FINAL = CONFIG["K_DENSE"], CONFIG["K_BM25"], CONFIG["RRF_K"], CONFIG["K_FINAL"]

def rrf_merge(*lists, k=RRF_K):
    scores, keep = {}, {}
    for lst in lists:
        for rank, t in enumerate(lst):
            key = t[:120]
            scores[key] = scores.get(key, 0.0) + 1.0 / (k + rank + 1)
            keep.setdefault(key, t)
    return [keep[key] for key in sorted(keep, key=lambda kk: scores[kk], reverse=True)]

def make_hybrid(m):
    emb = m.encode(all_texts, batch_size=64, normalize_embeddings=True, show_progress_bar=True).astype("float32")
    ix = faiss.IndexFlatIP(emb.shape[1]); ix.add(emb)
    def retrieve(german_query, k_final=K_FINAL):
        v = m.encode([german_query], normalize_embeddings=True).astype("float32")
        _, idx = ix.search(v, K_DENSE)
        dense  = [all_texts[i] for i in idx[0]]
        sparse = bm25.get_top_n(german_query.lower().split(" "), all_texts, n=K_BM25)
        fused  = rrf_merge(dense, sparse)
        sc = reranker.predict([[german_query, t] for t in fused])
        return [t for _, t in sorted(zip(sc, fused), key=lambda x: x[0], reverse=True)][:k_final]
    return retrieve

gt = pd.read_csv(find(CONFIG["GT_FILE"]))
ref_map = dict(zip(gt[CONFIG["GT_Q"]].astype(str).str.strip(), gt[CONFIG["GT_REF"]].astype(str)))

def retrieval_rows(m):
    retrieve = make_hybrid(m)
    rows = []
    for _, r in eval_df.iterrows():
        q = str(r[Q_EN]); ref = ref_map.get(q.strip())
        if not ref or str(ref).strip().upper() == "NOT_IN_CORPUS":
            continue
        rows.append({"question": q, "contexts": retrieve(str(r[Q_DE])), "ground_truth": ref})
    return rows

from datasets import Dataset, Features, Value, Sequence
from ragas import evaluate as ragas_evaluate
from ragas.metrics import context_precision, context_recall
from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper
from langchain_openai import ChatOpenAI
from langchain_huggingface import HuggingFaceEmbeddings

judge = LangchainLLMWrapper(ChatOpenAI(
    model=CONFIG["JUDGE_MODEL"], api_key=os.environ["OPENROUTER_API_KEY"],
    base_url="https://openrouter.ai/api/v1", temperature=0, max_retries=6))
judge_emb = LangchainEmbeddingsWrapper(HuggingFaceEmbeddings(model_name=CONFIG["BASE_MODEL"]))
feat = Features({"question": Value("string"), "contexts": Sequence(Value("string")), "ground_truth": Value("string")})

def run_ragas(rows):
    ds = Dataset.from_dict({
        "question":     [r["question"]     for r in rows],
        "contexts":     [r["contexts"]     for r in rows],
        "ground_truth": [r["ground_truth"] for r in rows],
    }, features=feat)
    out = ragas_evaluate(ds, metrics=[context_precision, context_recall], llm=judge, embeddings=judge_emb)
    return out.to_pandas()[["context_precision", "context_recall"]].mean().round(3)

print("base bge-m3 retrieval RAGAS:")
print(run_ragas(retrieval_rows(base_model)))
print("fine-tuned bge-m3 retrieval RAGAS:")
print(run_ragas(retrieval_rows(tuned_model)))

## 7. Interpretation

A genuine improvement appears as higher P@1, R@5, and MRR for the fine-tuned row on the unseen
proxy, and as higher Context Precision and Context Recall for the fine-tuned embedder on the RAGAS
retrieval evaluation. The proxy measures English to German alignment through parallel questions; the
RAGAS rows measure retrieval quality over the actual AWMF chunks and are the values comparable to
Table 4.2 and Table 4.3.

The training set is small and the contrastive signal is parallel questions rather than
query-to-chunk relevance, so an alignment gain does not automatically transfer to a corpus-retrieval
gain; the RAGAS rows are the decisive measurement. To use the fine-tuned embedder in the deployed
system, the retrieval module reads its query embedder from the BGE_EMBED_URL service seam, and a
fresh ingestion re-embeds the AWMF chunks with this model.